# TP 1B — Efficience : tester la marche aléatoire

**Séance 2, partie 2 · 50 minutes · en binômes · déposé en fin de partie**


### Objectifs

1. Tests de racine unitaire : ADF et KPSS, sur prix et rendements
2. **Ratio de variance de Lo–MacKinlay**, version robuste
3. Comparaison robuste / non robuste — et pourquoi l'écart
4. **Conclusion rédigée** — un verdict argumenté sur l'efficience

### Rappel du cours

> Une racine unitaire est **nécessaire mais très insuffisante** pour une marche
> aléatoire. Le ratio de variance teste RW3 — l'absence de corrélation des
> incréments — et c'est lui qui intéresse l'efficience.

::: Ce notebook est autonome
Il recharge le panier au démarrage : si vous avez manqué le TP 1A, vous pouvez
suivre sans rien reprendre.
:::

### Ce qui est noté

Le notebook exécuté **et** la cellule de conclusion en fin de TP — 25 % de la
note, déposée avant la fin de la partie.


---

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# retrouve outils_cours.py, que le notebook soit ouvert depuis 02-TP/ ou 02-TP/corriges/
for _cand in (Path.cwd(), *Path.cwd().parents[:3]):
    if (_cand / "outils_cours.py").exists():
        sys.path.insert(0, str(_cand))
        break
import outils_cours as oc

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (11, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": .3})
np.random.seed(2026)
print("Environnement prêt.")

from statsmodels.tsa.stattools import adfuller, kpss, acf

prix = oc.charger_panier()
r = oc.rendements_log(prix)
print(f"{len(r)} rendements, {prix.shape[1]} actifs.")

## 1. Racine unitaire : ADF et KPSS

Rappel : ADF a pour $H_0$ « racine unitaire », KPSS a pour $H_0$ « stationnarité ».
Les deux sont complémentaires.

In [ ]:
def tests_racine(serie, nom):
    adf_p = adfuller(serie.dropna(), autolag="AIC")[1]
    kpss_p = kpss(serie.dropna(), regression="c", nlags="auto")[1]
    return {
        "série": nom,
        "p(ADF)": round(adf_p, 4),
        "ADF rejette H0 ?": "oui" if adf_p < 0.05 else "non",
        "p(KPSS)": round(kpss_p, 4),
        "KPSS rejette H0 ?": "oui" if kpss_p < 0.05 else "non",
        "conclusion": (
            "stationnaire" if adf_p < 0.05 and kpss_p >= 0.05
            else "NON stationnaire" if adf_p >= 0.05 and kpss_p < 0.05
            else "ambigu — mémoire longue ou rupture ?"
        ),
    }

resultats = []
for c in prix.columns:
    resultats.append(tests_racine(np.log(prix[c]), f"log-prix {c}"))
for c in r.columns:
    resultats.append(tests_racine(r[c], f"rendements {c}"))
pd.DataFrame(resultats).set_index("série")

**Question 1.1** — Le résultat sur les log-prix est-il surprenant ?
Vaut-il spécifiquement pour les crypto-actifs ?

**Question 1.2** — Un camarade écrit : *« ADF ne rejette pas sur le Bitcoin,
donc le Bitcoin suit une marche aléatoire et le marché est efficient. »*
Relevez les **deux** erreurs de raisonnement.

*Vos réponses :*

>

## 2. Ratio de variance de Lo–MacKinlay

Le vrai test de la marche aléatoire. On utilise **impérativement** la version
robuste à l'hétéroscédasticité — au vu de la TP 1A, la version standard
rejetterait pour la mauvaise raison.

In [ ]:
lignes = []
for c in r.columns:
    for q in (2, 4, 8, 16):
        res = oc.variance_ratio(r[c], q, robuste=True)
        lignes.append({
            "actif": c, "q": q,
            "VR": round(res["VR"], 3),
            "z": round(res["z"], 2),
            "p": round(res["p_value"], 4),
            "rejet à 5 %": "oui" if res["p_value"] < 0.05 else "non",
        })
vr_table = pd.DataFrame(lignes).pivot(index="actif", columns="q",
                                      values=["VR", "p"]).round(3)
vr_table

**Question 2.1** — Pour quels actifs et quels horizons la marche aléatoire
est-elle rejetée ? Le rejet est-il concentré à horizon court ou long ?

**Question 2.2** — Comparez avec la version **non robuste**. Que se passe-t-il ?
Expliquez pourquoi.

In [ ]:
# À COMPLÉTER
# Comparer, pour q=4 et pour chaque actif, la p-value robuste et la p-value
# non robuste (argument robuste=False). Que constatez-vous, et pourquoi ?


**Question 2.3 — bonus, si vous avez terminé** — L'efficience du Bitcoin
a-t-elle changé après le lancement des ETF au comptant (janvier 2024) ?
Comparez $VR(q)$ avant et après.

In [ ]:
# À COMPLÉTER — BONUS, seulement si vous avez terminé le reste
# Découper la série BTC avant / après le 2024-01-11 et comparer VR(q).
# N'oubliez pas de commenter la PUISSANCE du test sur le sous-échantillon court.


---

## Votre conclusion

Sur votre actif, en une quinzaine de lignes :

1. Que disent ADF et KPSS — et **pourquoi cela ne suffit pas** à conclure sur
   l'efficience ?
2. Le ratio de variance robuste rejette-t-il, et à quels horizons $q$ ?
3. **Votre verdict** : prédictibilité exploitable, effet de microstructure, ou
   absence de preuve ? Et quelle est la **puissance** de votre test ?

> « ADF donne p = 0,43, donc marche aléatoire, donc marché efficient » empile les
> trois erreurs vues en cours. On attend l'inverse.

> **C'est cette cellule qui fait l'essentiel de la note.** On attend un jugement
> argumenté, pas une liste de résultats. Écrivez ci-dessous, en markdown.

*(votre réponse ici — remplacez cette ligne)*


---

## Dépôt — Tests d'efficience · 25 %

**Il vous reste 5 minutes.** Dans l'ordre :

1. `Kernel` → **Restart & Run All** — un notebook qui ne s'exécute pas n'est pas corrigé
2. `File` → `Save and Export Notebook As…` → **HTML**
3. Déposer sur l'ENT : `NOM1-NOM2_tests-d'efficience.html`

::: Vous n'avez pas fini ?
Déposez en l'état. Un notebook incomplet, exécutable et honnêtement conclu vaut
mieux qu'un notebook complet et faux.
:::
